# Payment Variation Study
#### Explore how Medicare pays different hospitals for same condition (through DRG)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from utils import load_clean

c:\Users\khpha\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
df = load_clean("data/cleaned_provider_drg_data.csv")
df.head()

,provider_ccn,provider_name,city,state,ruca,ruca_desc,drg_code,drg_desc,total_discharges,avg_charge,avg_payment,avg_medicare_payment,urbanicity,realization,markup
0,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,003,ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...,11,738478.636360,103236.272730,91218.181818,Metro,0.123522,8.095739
1,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,023,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,23,173562.086960,40220.217391,37634.565217,Metro,0.216836,4.611773
2,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,024,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,13,95613.307692,27305.461538,25644.307692,Metro,0.268209,3.728442
3,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,025,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...,22,182831.409090,31420.954545,23270.863636,Metro,0.127280,7.856666
4,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,038,EXTRACRANIAL PROCEDURES WITH CC,27,111318.555560,12183.925926,10532.222222,Metro,0.094613,10.569332


In [6]:
# With sepsis being the most common DRG, we can use the wealth of data points to then analyze the spread in payments 
top_drgs = (
    df.groupby(["drg_code", "drg_desc"], observed=True)["total_discharges"]
    .sum().nlargest(10).reset_index()
)
top_drgs

,drg_code,drg_desc,total_discharges
0,871,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,577119
1,291,HEART FAILURE AND SHOCK WITH MCC,304694
2,177,RESPIRATORY INFECTIONS AND INFLAMMATIONS WITH MCC,141351
3,193,SIMPLE PNEUMONIA AND PLEURISY WITH MCC,136649
4,872,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,103998
5,690,KIDNEY AND URINARY TRACT INFECTIONS WITHOUT MCC,91493
6,189,PULMONARY EDEMA AND RESPIRATORY FAILURE,89991
7,392,"ESOPHAGITIS, GASTROENTERITIS AND MISCELLANEOUS...",85970
8,280,"ACUTE MYOCARDIAL INFARCTION, DISCHARGED ALIVE ...",83841
9,689,KIDNEY AND URINARY TRACT INFECTIONS WITH MCC,79740


In [21]:
# Use p10 and p90 to analyze the spread, as this presents a more robust view of the data than just using min and max values, which can be skewed by outliers.
drg_spread = (
    df[df["drg_code"].isin(top_drgs["drg_code"])]
    .groupby(["drg_code", "drg_desc"], observed=True)["avg_payment"]
    .agg(
        n_hospitals="count", 
         median="median",
         p10=lambda s: s.quantile(0.10), 
         p90=lambda s: s.quantile(0.90)
    )
    .assign(p90_p10_ratio=lambda d: d["p90"] / d["p10"])
    .sort_values("p90_p10_ratio", ascending=False)
    .round(2)
    .reset_index()
)
display(drg_spread)

print(f"Sepsis (DRG 871) p90/p10 ratio: {drg_spread[drg_spread['drg_code'] == '871']['p90_p10_ratio'].values[0]}")

,drg_code,drg_desc,n_hospitals,median,p10,p90,p90_p10_ratio
0,280,"ACUTE MYOCARDIAL INFARCTION, DISCHARGED ALIVE ...",1825,13319.20,10776.00,19473.59,1.81
1,189,PULMONARY EDEMA AND RESPIRATORY FAILURE,2098,10780.89,8842.65,15652.38,1.77
2,871,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,2661,16181.67,13355.50,23576.37,1.77
3,291,HEART FAILURE AND SHOCK WITH MCC,2587,10959.42,8932.73,15748.21,1.76
4,872,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,2216,9126.25,7444.90,13045.67,1.75
5,690,KIDNEY AND URINARY TRACT INFECTIONS WITHOUT MCC,2133,7160.50,5854.17,10231.36,1.75
6,392,"ESOPHAGITIS, GASTROENTERITIS AND MISCELLANEOUS...",1962,7171.80,5824.92,10173.71,1.75
7,689,KIDNEY AND URINARY TRACT INFECTIONS WITH MCC,1945,9776.78,8108.38,13924.09,1.72
8,177,RESPIRATORY INFECTIONS AND INFLAMMATIONS WITH MCC,2332,13721.61,11347.86,19255.60,1.70
9,193,SIMPLE PNEUMONIA AND PLEURISY WITH MCC,2442,11253.30,9268.63,15494.11,1.67


Sepsis (DRG 871) p90/p10 ratio: 1.77


In [23]:
df.head()

,provider_ccn,provider_name,city,state,ruca,ruca_desc,drg_code,drg_desc,total_discharges,avg_charge,avg_payment,avg_medicare_payment,urbanicity,realization,markup
0,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,003,ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...,11,738478.636360,103236.272730,91218.181818,Metro,0.123522,8.095739
1,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,023,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,23,173562.086960,40220.217391,37634.565217,Metro,0.216836,4.611773
2,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,024,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,13,95613.307692,27305.461538,25644.307692,Metro,0.268209,3.728442
3,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,025,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...,22,182831.409090,31420.954545,23270.863636,Metro,0.127280,7.856666
4,010001,Southeast Health Medical Center,Dothan,AL,2.0,Metropolitan area high commuting: primary flow...,038,EXTRACRANIAL PROCEDURES WITH CC,27,111318.555560,12183.925926,10532.222222,Metro,0.094613,10.569332
